# Reproducible results
To get reproducible results from a trained LightGBM model or in the other word, the same model comes out every time you train with the same data, you need to control several sources of randomness. Here's what matters:
|Parameter|Controls|
|:-------|:--------|
|seed	|Which rows/features get randomly sampled|
|force_row_wise/force_col_wise|Which histogram-building algorithm runs|
|deterministic=True	|Forces fixed floating-point accumulation order within that algorithm|


### `seed` controls logical randomness

seed (and its sub-seeds like bagging_seed, feature_fraction_seed) controls decisions that come from a random number generator: which rows get sampled for bagging, which features get sampled for feature_fraction, how ties get broken in certain sampling steps, etc. If you fix these, the same rows and features get selected on every run — that part is fully reproducible on its own.

### What `deterministic=True` actually changes

It forces LightGBM to use a fixed, reproducible order for the summation/reduction steps in histogram construction, instead of whatever order threads happen to finish in. That's why the docs specifically say it needs to be paired with force_row_wise (or force_col_wise) — the deterministic accumulation logic is implemented against a fixed row/col layout; without pinning that, you'd still have ambiguity about which algorithm's accumulation path is even running.  

If accumulation path is not fixed, this will lead to an issue that **floating-point addition is not associative** — `(a + b) + c` can give a microscopically different result than `a + (b + c)`.  
When multiple threads each sum a chunk of rows and then combine their partial sums, the order in which those partial sums get combined depends on thread scheduling — which core finishes first, OS-level timing, etc.

**Deterministic parallel reduction**: threads still run in parallel (for speed), but the code combines their partial results using a fixed, predetermined structure — e.g., always combine chunk 1+2, then that result +chunk 3, in a fixed tree pattern based on chunk index, not on which thread happened to finish first. This structure doesn't depend on how many threads did the work, only on how the data was logically partitioned.

In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import math

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [2]:
RANDOM_STATE = 12345

In [3]:
housing_data = fetch_california_housing(as_frame=True)
housing_df = housing_data.frame
feature_cols = list(housing_data.feature_names)
target_cols = list(housing_data.target_names)
housing_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [4]:
X = housing_df[feature_cols].values
y = housing_df[target_cols].squeeze()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_valid, X_test, y_valid, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=RANDOM_STATE)

print('Train', X_train.shape, y_train.shape)
print('Valid', X_valid.shape, y_valid.shape)
print('Test', X_test.shape, y_test.shape)

Train (16512, 8) (16512,)
Valid (2064, 8) (2064,)
Test (2064, 8) (2064,)


In [5]:
train_data = lgb.Dataset(X_train, label=y_train, feature_name=feature_cols)
valid_data = lgb.Dataset(X_valid, label=y_valid, feature_name=feature_cols)
test_data = lgb.Dataset(X_test, label=y_test, feature_name=feature_cols)

In [6]:
params = {
    'boosting_type': 'gbdt',       # Gradient Boosting Decision Tree
    'objective': 'regression',     # L2 loss (Mean Squared Error); raw model output IS the prediction
    'seed': RANDOM_STATE,
    'deterministic': True,
    'force_row_wise': True
}

In [20]:
# Train with train data.
regressor1 = lgb.train(train_set=train_data, params=params)
model_info1 = regressor1.dump_model()

[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 16512, number of used features: 8
[LightGBM] [Info] Start training from score 2.064984


In [25]:
# Train with train data again.
regressor2 = lgb.train(train_set=train_data, params=params)
model_info2 = regressor2.dump_model()

[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 16512, number of used features: 8
[LightGBM] [Info] Start training from score 2.064984


In [26]:
# Train with valid data.
regressor3 = lgb.train(train_set=valid_data, params=params)
model_info3 = regressor3.dump_model()

[LightGBM] [Info] Total Bins 1837
[LightGBM] [Info] Number of data points in the train set: 2064, number of used features: 8
[LightGBM] [Info] Start training from score 2.072918


In [27]:
# Check reproducibility
print('model_info1 == model_info2 :', model_info1 == model_info2)

model_info1 == model_info2 : True


In [28]:
print('model_info1 == model_info3 :', model_info1 == model_info3)

model_info1 == model_info3 : False


In [29]:
print('model_info2 == model_info3 :', model_info2 == model_info3)

model_info2 == model_info3 : False
